# RAV-Med — Large-Scale Evaluation on PubMedQA

This notebook scales the RAV-Med retrieval-reliability evaluation from the
original n = 8 hand-picked instances up to a **configurable large sample**
(default 1,000+ query–passage pairs across the four CLEAN / DRIFT /
CONTRADICTION / NOISE conditions), using the **full PubMedQA release**
(`qiaojin/PubMedQA` on Hugging Face) instead of the 100-passage pilot subset.

**What this notebook does, end to end:**
1. Loads `pqa_labeled`, `pqa_unlabeled`, and `pqa_artificial`.
2. Builds a large passage pool and embeds it with the same BioBERT
   SentenceTransformer used in the paper.
3. For every evaluation query, programmatically constructs all **four**
   retrieval conditions (CLEAN / DRIFT / CONTRADICTION / NOISE) instead of
   relying on 2 hand-written passages per condition.
4. Scores every pair with the same RoBERTa-Large-MNLI verifier and computes
   the Evidence Reliability Score (ERS = S − C).
5. Reproduces Table IV (reliability results), Table V (ablation), and
   Table VI (threshold sensitivity) — but now over hundreds/thousands of
   instances per condition instead of one — and re-draws Fig. 2–4 with error
   bars / distributions.
6. Saves every table (CSV) and figure (PNG) to `/content/rav_med_outputs/`
   so you can paste updated numbers straight into the paper.

**Before you trust the numbers:** step 9 prints a random sample of
constructed pairs per condition so you can manually sanity-check that the
programmatic DRIFT / CONTRADICTION / NOISE construction actually produced
sensible pairs before running the full-scale pass. Read that cell's output
before reporting any results.

**Runtime:** on a free Colab T4, encoding + NLI scoring for ~1,000 queries ×
4 conditions (~4,000 pairs) takes roughly 15–30 minutes. Start with
`N_QUERIES = 200` to validate the whole pipeline (~3–5 min), then raise it.

---

### v2 fix (based on an actual N=200 run)

A first N=200 run surfaced a real methodology bug: PubMedQA's `question`
field is phrased as a yes/no *question* ("Does X reduce Y?"), but
RoBERTa-Large-MNLI is trained to score contradiction between two
**declarative** sentences. Feeding it a question as the hypothesis meant
the CONTRADICTION condition scored almost identically to CLEAN — the
negated passage and the true passage were both being judged against a
question mark the model can't really "agree with" or "contradict" in the
first place.

This version fixes it by using PubMedQA's own `long_answer` field (the
abstract's declarative conclusion sentence) as the claim fed to both the
retriever and the NLI verifier — exactly mirroring how the paper's own
worked example used a declarative claim ("Antibiotics cure viral
infections"), not a question. The CONTRADICTION passage now negates that
claim directly, instead of negating the first (often just background)
sentence of the retrieved context. DRIFT/NOISE candidate selection also
switched from fixed cosine-similarity cutoffs to **percentile-based**
thresholds computed per-query, since biomedical embeddings compress
similarity scores toward a higher baseline than the fixed 0.15/0.35 cutoffs
assumed.


## 1. Install dependencies

In [ ]:
!pip install -q datasets sentence-transformers transformers faiss-cpu accelerate


## 2. Config — start small, then scale up

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Start here for a quick end-to-end smoke test ---
N_QUERIES   = 200     # number of evaluation queries (RAV-Med paper used 2 per condition = 8 total)
                       # -> raise to 1000+ once step 9's spot-check looks good
POOL_SIZE   = 5000    # size of the candidate passage pool drawn from pqa_unlabeled + pqa_artificial
                       # (used as the source of DRIFT / NOISE passages)

EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"  # same as the paper
NLI_MODEL   = "roberta-large-mnli"                                       # same as the paper

# DRIFT/NOISE bands are now PERCENTILES of each query's own similarity
# distribution over the candidate pool, not fixed cosine cutoffs — fixed
# cutoffs (e.g. sim < 0.15) turned out to be miscalibrated for biomedical
# embeddings, which compress similarity scores toward a higher baseline.
DRIFT_PERCENTILE_BAND = (40, 70)   # DRIFT = candidates between the 40th-70th percentile of similarity
NOISE_PERCENTILE_MAX  = 10         # NOISE = candidates at or below the 10th percentile of similarity
SEARCH_K = 3000                    # how many candidates to pull from FAISS per query before filtering

BATCH_SIZE = 32
DEVICE = "cuda"  # Colab: Runtime > Change runtime type > T4 GPU


## 3. Load PubMedQA

`pqa_labeled` gives us expert-annotated (question, true context, yes/no/maybe
decision) triples to use as evaluation queries. `pqa_unlabeled` and
`pqa_artificial` supply the large passage pool we draw DRIFT and NOISE
distractors from — this is what lets us go from 100 curated passages to a
pool of thousands.


In [ ]:
from datasets import load_dataset

print("Loading pqa_labeled ...")
pqa_labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

print("Loading pqa_unlabeled ...")
pqa_unlabeled = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")

print("Loading pqa_artificial (streaming a slice — it's 211k rows) ...")
pqa_artificial = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

print(len(pqa_labeled), len(pqa_unlabeled), len(pqa_artificial))


## 4. Flatten contexts and build the passage pool

Each PubMedQA context is a *list* of abstract sentences. We join them into
one passage string per example, and keep both the human-readable
`question` (for display/reporting only) and the declarative `claim` (the
**first sentence** of `long_answer` — the actual text we feed to the
retriever and the NLI verifier).

We restrict `claim` to one sentence, not the full (often multi-sentence)
`long_answer`, based on a bug found in an earlier run: when only the first
sentence of a multi-sentence claim was negated but the *full* claim stayed
as the NLI hypothesis, the premise only contradicted part of the
hypothesis and the contradiction score came out artificially low
(0.35 mean vs. 0.61 for single-sentence claims). Scoping both sides to the
same single sentence removes that mismatch entirely.

We also build a large distractor pool from unlabeled + artificial examples.


In [ ]:
import re

def join_context(example):
    return " ".join(example["context"]["contexts"])

def first_sentence_of(text):
    text = text.strip()
    if not text:
        return text
    return re.split(r"(?<=[.!?])\s+", text)[0]

# Evaluation queries: question (display only), claim (declarative, single
# sentence — used for retrieval + NLI), true_context, true_decision, pubid
labeled_records = []
for ex in pqa_labeled:
    if not ex["long_answer"] or not ex["long_answer"].strip():
        continue  # skip the rare row with no usable declarative claim
    labeled_records.append({
        "pubid": ex["pubid"],
        "question": ex["question"],
        "claim": first_sentence_of(ex["long_answer"]),
        "context": join_context(ex),
        "decision": ex["final_decision"],  # yes / no / maybe
    })

random.shuffle(labeled_records)
eval_queries = labeled_records[:N_QUERIES]
print(f"Using {len(eval_queries)} evaluation queries out of {len(labeled_records)} available in pqa_labeled")

# Distractor pool: sampled from unlabeled + artificial, deduplicated by pubid
pool_records = []
for ex in pqa_unlabeled.shuffle(seed=SEED).select(range(min(POOL_SIZE // 2, len(pqa_unlabeled)))):
    pool_records.append({"pubid": ex["pubid"], "context": join_context(ex)})
for ex in pqa_artificial.shuffle(seed=SEED).select(range(min(POOL_SIZE // 2, len(pqa_artificial)))):
    pool_records.append({"pubid": ex["pubid"], "context": join_context(ex)})

# also fold in the labeled contexts themselves so CLEAN passages are retrievable
for r in labeled_records:
    pool_records.append({"pubid": r["pubid"], "context": r["context"]})

seen = set()
deduped_pool = []
for r in pool_records:
    if r["pubid"] not in seen:
        seen.add(r["pubid"])
        deduped_pool.append(r)
pool_records = deduped_pool
print(f"Passage pool size: {len(pool_records)}")


## 5. Embed the pool and build a FAISS index

Same embedding model as the paper (BioBERT SentenceTransformer). We
L2-normalize so inner product = cosine similarity, and use
`IndexFlatIP` so similarity search matches the paper's cosine-similarity
formulation exactly (rather than L2 distance).


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

pool_texts = [r["context"] for r in pool_records]
pool_embeddings = embedder.encode(
    pool_texts, batch_size=BATCH_SIZE, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)

index = faiss.IndexFlatIP(pool_embeddings.shape[1])
index.add(pool_embeddings.astype("float32"))
print("FAISS index size:", index.ntotal)


## 6. Rule-based contradiction generator

Rather than mining "opposite polarity" documents (which are not
necessarily *about* the same claim, and so are not reliable contradictions),
we generate a contradiction the same way the paper's single hand-written
example did — by negating the claim itself. This is simple, auditable, and
guarantees the CONTRADICTION passage genuinely conflicts with the query,
rather than just being topically different.

### v3 fix (based on an actual N=200 run)

That first version had two real bugs, found by inspecting failed rows
(cases where the NLI model scored the "contradiction" near zero, i.e. it
didn't register as a contradiction at all — about a third of all
CONTRADICTION rows):

1. **Word-boundary bug.** Plain substring matching let a short keyword like
   `"cause"` match *inside* an unrelated word like `"be`**`cause`**`"`,
   corrupting the sentence into nonsense. Fixed by matching on `\b`-bounded
   regex patterns instead of `str.find`.
2. **Double-negation cancellation.** Many PubMedQA conclusions are already
   negative sentences ("was **not** associated with...", "**no** evidence
   that..."). Blindly inserting another negative on top ("It is not true
   that ... was not associated with...") is *logically* still a valid
   negation, but it's so lexically similar to the original and so
   awkwardly double-negated that the NLI model consistently failed to flag
   it as a contradiction. The fix detects existing negation cues first and,
   if found, **strips/flips** them to produce a clean affirmative statement,
   instead of piling on a second negation.

This dropped the failure rate (contradiction score < 0.1) from 33.5% to
5.5% on a real N=200 run. Inspecting the *remaining* 11 failures surfaced
two more, smaller patterns fixed below in **v4**:

3. **Noun/verb ambiguity.** Words like "cause" and "increase" are often
   nouns in PubMedQA text ("a possible **cause of** unsteadiness"), but the
   v3 patterns always treated them as verbs, producing broken grammar like
   *"a possible **does not cause** of unsteadiness."* Fixed with a negative
   lookahead so these patterns skip the noun-phrase form (`cause` followed
   by `of`/`for`, `increase` followed by `in`/`of`, etc.) and fall back to
   the generic wrapper instead of generating nonsense.
4. **Incomplete strip-pattern coverage.** Some already-negative phrasings
   ("unusual but **not rare**", "**no alteration** in practice", "**would
   not have** missed") weren't covered by any specific strip pattern, so
   they fell through to the double-negation-prone wrapper — the exact
   problem v3 was supposed to fix. Added specific patterns for these, plus
   a generic last-resort rule (strip a bare standalone "not"/"no") that
   fires *before* the wrapper, so double-negation is now very hard to
   reach even for phrasings we didn't anticipate. Also added a cleanup
   step for dangling correlatives — flipping "does not affect ... **nor**
   alter" only on the "does not" leaves a stray "nor" that reads as
   nonsense; it's now converted to "and" when this happens.

Extend either pattern list with more domain terms if you want more lexical
variety; the generic wrapper fallback only fires when nothing in either
list matches.


In [ ]:
import re

# If the claim ALREADY contains a negation, flip it to affirmative
# (checked first) rather than stacking a second negation on top of it.
# Ordered from most specific to most generic — the last two entries are a
# last-resort catch-all so double-negation is hard to reach even for
# phrasings not explicitly covered above them.
NEGATION_STRIP_PATTERNS = [
    (r"\bwas not associated with\b", "was associated with"),
    (r"\bwere not associated with\b", "were associated with"),
    (r"\bis not associated with\b", "is associated with"),
    (r"\bare not associated with\b", "are associated with"),
    (r"\bwould not have\b", "would have"),
    (r"\bdid not have\b", "did have"),
    (r"\bno evidence that\b", "clear evidence that"),
    (r"\bno evidence of\b", "clear evidence of"),
    (r"\bno significant difference\b", "a significant difference"),
    (r"\bno significant association\b", "a significant association"),
    (r"\bno alteration in\b", "a clear alteration in"),
    (r"\bno alteration\b", "a clear alteration"),
    (r"\bno benefit\b", "a clear benefit"),
    (r"\bno association\b", "a strong association"),
    (r"\bnot rare\b", "rare"),
    (r"\bnot necessarily\b", "certainly"),
    (r"\bfails to\b", "succeeds in"),
    (r"\bfailed to\b", "succeeded in"),
    (r"\bwithout\b", "with"),
    (r"\black of\b", "presence of"),
    (r"\bnone\b", "some"),
    (r"\bneither\b", ""),
    (r"\bdoes not\b", "does"),
    (r"\bdo not\b", "do"),
    (r"\bdid not\b", "did"),
    (r"\bcan not\b", "can"),
    (r"\bcannot\b", "can"),
    (r"\bis not\b", "is"),
    (r"\bare not\b", "are"),
    (r"\bwas not\b", "was"),
    (r"\bwere not\b", "were"),
    (r"\bhas not\b", "has"),
    (r"\bhave not\b", "have"),
    # generic last-resort: soften/remove a bare standalone negator rather
    # than falling through to the double-negation-prone wrapper
    (r"\bnot\b", ""),
    (r"\bno\b", "significant"),
]

# If the claim is affirmative, ADD a negation (word-boundary safe). Terms
# that are commonly nouns in this domain ("a possible cause of...", "an
# increase in...") use a negative lookahead so we skip the noun-phrase
# form rather than produce broken grammar.
NEGATION_ADD_PATTERNS = [
    (r"\bincreases\b(?!\s+(?:in|of))", "does not increase"), (r"\bincrease\b(?!\s+(?:in|of))", "does not increase"),
    (r"\bdecreases\b(?!\s+(?:in|of))", "does not decrease"), (r"\bdecrease\b(?!\s+(?:in|of))", "does not decrease"),
    (r"\breduces\b(?!\s+(?:in|of))", "does not reduce"), (r"\breduce\b(?!\s+(?:in|of))", "does not reduce"),
    (r"\bimproves\b(?!\s+(?:in|of))", "does not improve"), (r"\bimprove\b(?!\s+(?:in|of))", "does not improve"),
    (r"\bprevents\b(?!\s+(?:of|from))", "does not prevent"), (r"\bprevent\b(?!\s+(?:of|from))", "does not prevent"),
    (r"\bcauses\b(?!\s+(?:of|for))", "does not cause"), (r"\bcause\b(?!\s+(?:of|for))", "does not cause"),
    (r"\bis effective\b", "is not effective"), (r"\bare effective\b", "are not effective"),
    (r"\bis safe\b", "is not safe"), (r"\bare safe\b", "are not safe"),
    (r"\bis recommended\b", "is contraindicated"), (r"\bare recommended\b", "are contraindicated"),
    (r"\bcures\b(?!\s+(?:for|of))", "does not cure"), (r"\bcure\b(?!\s+(?:for|of))", "does not cure"),
    (r"\bhelps\b(?!\s+(?:for|with))", "does not help"), (r"\bhelp\b(?!\s+(?:for|with))", "does not help"),
    (r"\bbenefits\b(?!\s+(?:for|of|in))", "does not benefit"), (r"\bbenefit\b(?!\s+(?:for|of|in))", "does not benefit"),
    (r"\bassociated with\b", "not associated with"),
]

NEGATION_CUE = re.compile(
    r"\b(not|no|without|cannot|can\'t|fails to|failed to|lack of|none|neither|nor)\b",
    re.IGNORECASE,
)

def negate_claim(text: str) -> str:
    """Return a confidently-negated (or, if already negative, a flipped
    affirmative) version of `text`. Expects `text` to already be a single
    sentence (see first_sentence_of in step 4) so premise and hypothesis
    stay scoped to the same claim."""
    sentence = text.strip()
    patterns = NEGATION_STRIP_PATTERNS if NEGATION_CUE.search(sentence) else NEGATION_ADD_PATTERNS
    for pattern, repl in patterns:
        new_sentence, n = re.subn(pattern, repl, sentence, count=1, flags=re.IGNORECASE)
        if n > 0:
            # clean up a dangling correlative left behind by a partial flip
            # (e.g. "does not affect X nor alter Y" -> "does affect X nor alter Y")
            if " nor " in new_sentence and "neither" not in new_sentence.lower():
                new_sentence = re.sub(r"\bnor\b", "and", new_sentence, flags=re.IGNORECASE)
            return re.sub(r"\s{2,}", " ", new_sentence).strip()
    return f"It is not true that {sentence[0].lower() + sentence[1:]}"

# quick sanity check — includes already-negative and noun-ambiguous examples
print(negate_claim("Insulin helps regulate blood glucose levels."))
print(negate_claim("Antibiotics are ineffective against viral infections."))
print(negate_claim("Statins reduce cardiovascular risk in high-risk patients."))
print(negate_claim("Metformin was not associated with improved outcomes."))
print(negate_claim("There appears to be no benefit of early intervention."))
print(negate_claim("These results do not support the view that AA women are at greater risk for obesity."))
print(negate_claim("This is unusual but not rare."))
print(negate_claim("A possible cause of unsteadiness is orthostatic myoclonus."))


## 7. Construct CLEAN / DRIFT / CONTRADICTION / NOISE for every query

For each evaluation query, retrieval and NLI scoring are both driven by the
**declarative claim** (`long_answer`), not the raw question — see the v2
fix note in the intro.

- **CLEAN** — the query's own true context passage.
- **DRIFT** — a *different real* passage from the pool whose similarity to
  the claim falls in the 40th–70th percentile of that query's own
  candidate-similarity distribution (semantically related, not the right
  answer).
- **CONTRADICTION** — the claim itself, negated (on-topic by construction,
  factually conflicting).
- **NOISE** — a real passage from the pool at or below the 10th percentile
  of that query's candidate-similarity distribution (largely unrelated).

Percentiles are computed **per query** rather than as fixed cosine cutoffs,
since biomedical embeddings compress similarity scores toward a higher
baseline than general-domain text — a fixed threshold that works for one
embedding model/domain will misfire on another.


In [ ]:
def build_conditions_for_query(claim_text, true_context, true_pubid, k=SEARCH_K):
    k = min(k, index.ntotal)
    q_emb = embedder.encode([claim_text], normalize_embeddings=True, convert_to_numpy=True)
    sims, idxs = index.search(q_emb.astype("float32"), k)
    sims, idxs = sims[0], idxs[0]

    mask = np.array([pool_records[i]["pubid"] != true_pubid for i in idxs])
    sims_f, idxs_f = sims[mask], idxs[mask]

    if len(sims_f) < 20:
        # degenerate case (tiny pool) — just fall back to random passages
        r1, r2 = random.sample(pool_records, 2)
        drift_passage, noise_passage = r1["context"], r2["context"]
    else:
        drift_lo, drift_hi = np.percentile(sims_f, DRIFT_PERCENTILE_BAND)
        noise_ceiling = np.percentile(sims_f, NOISE_PERCENTILE_MAX)

        drift_candidates = [pool_records[i] for i, s in zip(idxs_f, sims_f) if drift_lo <= s <= drift_hi]
        noise_candidates = [pool_records[i] for i, s in zip(idxs_f, sims_f) if s <= noise_ceiling]

        # both bands are defined by percentiles of sims_f, so they can only
        # come up empty due to ties/rounding — fall back to the nearest rank
        if not drift_candidates:
            mid = len(idxs_f) // 2
            drift_candidates = [pool_records[idxs_f[mid]]]
        if not noise_candidates:
            noise_candidates = [pool_records[idxs_f[-1]]]  # least-similar candidate found

        drift_passage = random.choice(drift_candidates)["context"]
        noise_passage = random.choice(noise_candidates)["context"]

    contradiction_passage = negate_claim(claim_text)

    return {
        "CLEAN": true_context,
        "DRIFT": drift_passage,
        "CONTRADICTION": contradiction_passage,
        "NOISE": noise_passage,
    }


## 8. Build the full evaluation set

This runs the condition-construction logic over all `N_QUERIES` queries and
produces one row per (query, condition) pair — 4 rows per query.


In [ ]:
rows = []
for r in eval_queries:
    conditions = build_conditions_for_query(r["claim"], r["context"], r["pubid"])
    for cond_name, passage_text in conditions.items():
        rows.append({
            "pubid": r["pubid"],
            "question": r["question"],   # display only — not fed to any model
            "claim": r["claim"],          # declarative — this is what similarity/NLI actually score
            "condition": cond_name,
            "passage": passage_text,
        })

import pandas as pd
eval_df = pd.DataFrame(rows)
print(eval_df.shape)
eval_df.head(8)


## 9. ⚠️ Manual spot-check before scaling up

Read these examples. If DRIFT passages look like NOISE, or CONTRADICTION
sentences look awkward/nonsensical, tighten `DRIFT_PERCENTILE_BAND` /
`NOISE_PERCENTILE_MAX` or extend `NEGATION_PAIRS` before running the full
pipeline on 1,000+ queries. The `claim` column is what's actually scored —
check that it reads as a plausible declarative fact, not a fragment.


In [ ]:
for cond in ["CLEAN", "DRIFT", "CONTRADICTION", "NOISE"]:
    sample = eval_df[eval_df.condition == cond].sample(3, random_state=SEED)
    print(f"\n=== {cond} ===")
    for _, row in sample.iterrows():
        print("Question (display only):", row.question)
        print("Claim (scored):", row.claim)
        print("Passage:", row.passage[:220], "...\n")


## 10. Load the NLI verifier and scoring functions

Same RoBERTa-Large-MNLI verifier as the paper. We read the contradiction
index off `model.config.id2label` rather than hard-coding it, since label
order isn't guaranteed to be identical across checkpoint versions.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE)
nli_model.eval()

contradiction_idx = [i for i, lbl in nli_model.config.id2label.items()
                      if lbl.lower().startswith("contra")][0]
print("Contradiction label index:", contradiction_idx, nli_model.config.id2label)

@torch.no_grad()
def contradiction_scores(premises, hypotheses, batch_size=BATCH_SIZE):
    scores = []
    for i in range(0, len(premises), batch_size):
        batch_p = premises[i:i + batch_size]
        batch_h = hypotheses[i:i + batch_size]
        enc = tokenizer(batch_p, batch_h, truncation=True, padding=True,
                         max_length=256, return_tensors="pt").to(DEVICE)
        logits = nli_model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        scores.extend(probs[:, contradiction_idx].cpu().tolist())
    return scores

def cosine_sim(a_texts, b_texts, batch_size=BATCH_SIZE):
    a_emb = embedder.encode(a_texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)
    b_emb = embedder.encode(b_texts, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True)
    return (a_emb * b_emb).sum(axis=1)


## 11. Score every pair: Similarity, Contradiction, ERS

Premise = retrieved passage, hypothesis = the declarative **claim**
(`long_answer`) — matching the paper's NLI framing (Section III.D), and
critically *not* the raw interrogative question (see the v2 fix note).


In [ ]:
eval_df["similarity"] = cosine_sim(eval_df["passage"].tolist(), eval_df["claim"].tolist())
eval_df["contradiction"] = contradiction_scores(eval_df["passage"].tolist(), eval_df["claim"].tolist())
eval_df["ERS"] = eval_df["similarity"] - eval_df["contradiction"]  # alpha = beta = 1

eval_df.to_csv("rav_med_full_results.csv", index=False)
eval_df.head(8)


## 11b. Diagnostic: did the CONTRADICTION negation actually register?

A healthy run should show most CONTRADICTION rows scoring **high**
contradiction (the negation was detected) and very few near zero (the
negation silently failed — indistinguishable from CLEAN). Earlier runs
surfaced real bugs this way; re-check this cell any time you change
`NEGATION_STRIP_PATTERNS` / `NEGATION_ADD_PATTERNS` or raise `N_QUERIES`.


In [ ]:
contra_scores = eval_df.loc[eval_df.condition == "CONTRADICTION", "contradiction"]
failed_frac = (contra_scores < 0.1).mean()
strong_frac = (contra_scores > 0.9).mean()

print(f"CONTRADICTION rows: {len(contra_scores)}")
print(f"  scoring < 0.1 (negation likely failed): {failed_frac:.1%}")
print(f"  scoring > 0.9 (negation clearly detected): {strong_frac:.1%}")
print(f"  mean contradiction: {contra_scores.mean():.3f}")

if failed_frac > 0.15:
    print("\n⚠️  More than 15% of CONTRADICTION rows look like failed negations.")
    print("    Inspect a few of them before trusting the aggregate tables:")
    display(eval_df[(eval_df.condition == "CONTRADICTION") & (eval_df.contradiction < 0.1)]
            [["claim", "passage", "contradiction"]].head(5))


## 12. Table IV equivalent — reliability results at scale (mean ± std)

In [ ]:
summary = eval_df.groupby("condition")[["similarity", "contradiction", "ERS"]].agg(["mean", "std", "count"])
summary = summary.reindex(["CLEAN", "DRIFT", "CONTRADICTION", "NOISE"])
print(summary)
summary.to_csv("table_iv_scaled_reliability_results.csv")


## 13. Table V equivalent — ablation at scale (Full vs. Similarity-only vs. Contradiction-only)


In [ ]:
ablation = eval_df.copy()
ablation["Full_RAV-Med"] = ablation["similarity"] - ablation["contradiction"]
ablation["Similarity_Only"] = ablation["similarity"]
ablation["Contradiction_Only"] = -ablation["contradiction"]

ablation_summary = ablation.groupby("condition")[
    ["Full_RAV-Med", "Similarity_Only", "Contradiction_Only"]
].mean().reindex(["CLEAN", "DRIFT", "CONTRADICTION", "NOISE"])

print(ablation_summary)
ablation_summary.to_csv("table_v_scaled_ablation.csv")

overestimation_pct = (
    (ablation_summary.loc["CONTRADICTION", "Similarity_Only"] -
     ablation_summary.loc["CONTRADICTION", "Full_RAV-Med"])
    / abs(ablation_summary.loc["CONTRADICTION", "Full_RAV-Med"]) * 100
)
print(f"Similarity-only overestimation under CONTRADICTION at scale: {overestimation_pct:.1f}%")


## 14. Table VI equivalent — threshold sensitivity + macro F1/precision/recall

This is the key upgrade over the original 8-instance table: with hundreds
of labeled instances per condition we can report real classification
metrics (macro precision/recall/F1 of SAFE/RISKY/UNSAFE vs. the
programmatic condition label), not just a single classification per
condition.


In [ ]:
from sklearn.metrics import classification_report

def classify(ers, T):
    if ers >= T:
        return "SAFE"
    elif ers >= T - 0.2:
        return "RISKY"
    else:
        return "UNSAFE"

# Ground-truth safety label implied by construction:
#   CLEAN -> SAFE, DRIFT -> RISKY, CONTRADICTION/NOISE -> UNSAFE
condition_to_expected = {"CLEAN": "SAFE", "DRIFT": "RISKY",
                          "CONTRADICTION": "UNSAFE", "NOISE": "UNSAFE"}
eval_df["expected_label"] = eval_df["condition"].map(condition_to_expected)

threshold_rows = []
for T in [0.0, 0.2, 0.4, 0.6]:
    eval_df[f"pred_T{T}"] = eval_df["ERS"].apply(lambda e: classify(e, T))
    report = classification_report(
        eval_df["expected_label"], eval_df[f"pred_T{T}"],
        output_dict=True, zero_division=0,
    )
    threshold_rows.append({
        "T": T,
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
    })
    print(f"--- T = {T} ---")
    print(classification_report(eval_df["expected_label"], eval_df[f"pred_T{T}"], zero_division=0))

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.to_csv("table_vi_scaled_threshold_metrics.csv", index=False)
threshold_summary


## 15. Figures — Fig. 2 / Fig. 3 / Fig. 4 equivalents, now with distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

order = ["CLEAN", "DRIFT", "CONTRADICTION", "NOISE"]

# Fig. 2 equivalent: mean ERS by condition with error bars
plt.figure(figsize=(6, 4))
means = eval_df.groupby("condition")["ERS"].mean().reindex(order)
stds = eval_df.groupby("condition")["ERS"].std().reindex(order)
plt.bar(order, means, yerr=stds, capsize=4)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("ERS")
plt.title(f"Evidence Reliability Score by Condition (n={len(eval_df)//4} queries/condition)")
plt.tight_layout()
plt.savefig("fig2_scaled_ers_by_condition.png", dpi=150)
plt.show()

# Fig. 3 equivalent: similarity vs contradiction scatter, colored by condition
plt.figure(figsize=(6, 5))
for cond in order:
    sub = eval_df[eval_df.condition == cond]
    plt.scatter(sub["similarity"], sub["contradiction"], label=cond, alpha=0.5, s=12)
plt.xlabel("Similarity Score")
plt.ylabel("Contradiction Score")
plt.title("Similarity vs. Contradiction Across Retrieval Conditions (scaled)")
plt.legend()
plt.tight_layout()
plt.savefig("fig3_scaled_similarity_vs_contradiction.png", dpi=150)
plt.show()

# Extra: ERS distribution (box/violin) per condition — not possible at n=1
plt.figure(figsize=(6, 4))
sns.boxplot(data=eval_df, x="condition", y="ERS", order=order)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("ERS Distribution by Condition (scaled evaluation)")
plt.tight_layout()
plt.savefig("fig_extra_ers_distribution_boxplot.png", dpi=150)
plt.show()


## 16. Package everything for download

In [ ]:
import shutil, os

os.makedirs("rav_med_outputs", exist_ok=True)
for fname in [
    "rav_med_full_results.csv",
    "table_iv_scaled_reliability_results.csv",
    "table_v_scaled_ablation.csv",
    "table_vi_scaled_threshold_metrics.csv",
    "fig2_scaled_ers_by_condition.png",
    "fig3_scaled_similarity_vs_contradiction.png",
    "fig_extra_ers_distribution_boxplot.png",
]:
    if os.path.exists(fname):
        shutil.copy(fname, os.path.join("rav_med_outputs", fname))

shutil.make_archive("rav_med_outputs", "zip", "rav_med_outputs")
print("Saved rav_med_outputs.zip — download it from the Colab file browser (left sidebar).")

try:
    from google.colab import files
    files.download("rav_med_outputs.zip")
except ImportError:
    pass


## Notes on what to check / adjust before reporting these numbers

- **Spot-check (step 9) is mandatory, not optional.** The DRIFT/NOISE
  percentile bands (`DRIFT_PERCENTILE_BAND`, `NOISE_PERCENTILE_MAX`) are
  reasonable starting heuristics, not validated thresholds — inspect real
  examples and tighten them if a "DRIFT" passage reads as unrelated, or a
  "NOISE" passage happens to be topically close.
- **Both retrieval and NLI scoring use the declarative `claim`
  (`long_answer`), never the raw `question`.** If you extend this notebook,
  keep that separation — question-form hypotheses are a known failure mode
  for MNLI-style verifiers (this is exactly the bug the v2 fix addresses;
  don't reintroduce it in Table VI's `expected_label` logic or anywhere
  else that touches the NLI model).
- **The CONTRADICTION construction is rule-based negation of the claim**,
  which guarantees a genuine logical conflict but produces less lexically
  varied contradictions than a human writer would. If you have budget for
  it, an LLM-based paraphrase-and-negate step (e.g., one call per query to
  a hosted model) would diversify phrasing — not included here since it
  adds an external API dependency and cost.
- **CONTRADICTION similarity scores will run high (often 0.7–0.9), higher
  than DRIFT/NOISE and higher than the paper's single hand-written example
  (0.58).** This isn't a bug to chase further — sentence embeddings are
  well documented to be weak at capturing negation lexically, so a
  passage that is a near-verbatim negation of the claim will still look
  "topically identical" to a bi-encoder. Because ERS = S − C, this means
  CONTRADICTION's *mean ERS* can end up higher than DRIFT/NOISE even when
  its contradiction score is correctly the highest of the four — report
  this as an observed property of the similarity component, rather than
  forcing the aggregate ERS ranking to match the original n=8 pilot.
- **Use cell 11b after every run**, especially after changing
  `N_QUERIES`, the negation pattern lists, or the embedding/NLI model. It
  flags when negations are silently failing (contradiction score near
  zero) before you get as far as the summary tables.
- **The "expected label" mapping in step 14** (CLEAN→SAFE, DRIFT→RISKY,
  CONTRADICTION/NOISE→UNSAFE) is an evaluation convenience so you can
  compute macro F1 — state this construction explicitly in the paper's
  methods section rather than presenting it as independently-sourced
  ground truth.
- Once `N_QUERIES` is raised to 1,000+, expect the NLI-scoring cell
  (step 11) to be the slowest step — batch size 32 on a T4 is a reasonable
  default; drop to 16 if you hit an out-of-memory error.
- **Re-run steps 6 onward after changing `N_QUERIES` or `POOL_SIZE`** —
  they depend on the pool and eval-query sample built in step 4.
